# 📧 Spam Email Classifier
**Author:** Aladdin  
**Goal:** Build a machine learning model to classify emails as spam or not spam using NLP techniques.

---
### Techniques Used
- Text preprocessing & cleaning
- TF-IDF Vectorization
- Naive Bayes (baseline)
- Logistic Regression (best model)
- Model evaluation: Accuracy, Precision, Recall, F1-Score, Confusion Matrix

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

import re
import string
import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully ✅')

## 2. Load Dataset
> Dataset: [SMS Spam Collection](https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset)  
> Place the CSV file in the `../data/` folder as `spam.csv`

In [ ]:
df = pd.read_csv('../data/spam.csv', encoding='latin-1')

# Keep only relevant columns
df = df[['v1', 'v2']]
df.columns = ['label', 'text']

print(f'Dataset shape: {df.shape}')
print(f'\nClass distribution:')
print(df['label'].value_counts())
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Class distribution plot
plt.figure(figsize=(6, 4))
sns.countplot(x='label', data=df, palette='Set2')
plt.title('Spam vs Ham Distribution')
plt.xlabel('Label')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('../data/class_distribution.png', dpi=150)
plt.show()

# Text length analysis
df['text_length'] = df['text'].apply(len)
plt.figure(figsize=(8, 4))
df.groupby('label')['text_length'].plot.hist(bins=50, alpha=0.6, legend=True)
plt.title('Text Length Distribution by Class')
plt.xlabel('Text Length')
plt.tight_layout()
plt.show()

## 4. Text Preprocessing

In [ ]:
def preprocess_text(text):
    """Clean and normalize email/SMS text."""
    # Lowercase
    text = text.lower()
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Remove extra whitespace
    text = text.strip()
    return text

df['clean_text'] = df['text'].apply(preprocess_text)

# Preview
print('Original:', df['text'][0])
print('Cleaned: ', df['clean_text'][0])

## 5. Feature Engineering — TF-IDF Vectorization

In [ ]:
# Encode labels
df['label_encoded'] = df['label'].map({'ham': 0, 'spam': 1})

X = df['clean_text']
y = df['label_encoded']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# TF-IDF Vectorizer
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english')
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f'Training samples : {X_train_tfidf.shape[0]}')
print(f'Test samples     : {X_test_tfidf.shape[0]}')
print(f'TF-IDF features  : {X_train_tfidf.shape[1]}')

## 6. Model Training & Evaluation

In [ ]:
def evaluate_model(name, model, X_test, y_test):
    """Train, predict, and print evaluation metrics."""
    y_pred = model.predict(X_test)
    print(f'\n=== {name} ===')
    print(f'Accuracy  : {accuracy_score(y_test, y_pred):.4f}')
    print(f'Precision : {precision_score(y_test, y_pred):.4f}')
    print(f'Recall    : {recall_score(y_test, y_pred):.4f}')
    print(f'F1-Score  : {f1_score(y_test, y_pred):.4f}')
    print('\nClassification Report:')
    print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))
    return y_pred

# --- Naive Bayes ---
nb = MultinomialNB()
nb.fit(X_train_tfidf, y_train)
nb_preds = evaluate_model('Naive Bayes', nb, X_test_tfidf, y_test)

# --- Logistic Regression ---
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_tfidf, y_train)
lr_preds = evaluate_model('Logistic Regression', lr, X_test_tfidf, y_test)

## 7. Confusion Matrix Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, preds, title in zip(
    axes,
    [nb_preds, lr_preds],
    ['Naive Bayes', 'Logistic Regression']
):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Ham', 'Spam'], yticklabels=['Ham', 'Spam'])
    ax.set_title(f'{title} — Confusion Matrix')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig('../data/confusion_matrices.png', dpi=150)
plt.show()

## 8. Save the Best Model

In [ ]:
import pickle
import os

os.makedirs('../models', exist_ok=True)

# Save Logistic Regression model and TF-IDF vectorizer
with open('../models/spam_classifier.pkl', 'wb') as f:
    pickle.dump(lr, f)

with open('../models/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

print('Model and vectorizer saved to ../models/ ✅')

## 9. Predict on New Input

In [ ]:
def predict_spam(text):
    """Predict whether a given text is spam or ham."""
    cleaned = preprocess_text(text)
    vectorized = tfidf.transform([cleaned])
    prediction = lr.predict(vectorized)[0]
    probability = lr.predict_proba(vectorized)[0][prediction]
    label = '🚨 SPAM' if prediction == 1 else '✅ HAM'
    print(f'Text      : {text}')
    print(f'Prediction: {label} ({probability*100:.2f}% confidence)\n')

# Test examples
predict_spam("Congratulations! You've won a free iPhone. Click here to claim now!")
predict_spam("Hey, are we still meeting tomorrow at 10am?")
predict_spam("URGENT: Your bank account has been suspended. Verify now at this link.")